# 07 · Spatial EMFI attack map

CNC positioning (nb 06) + the attack→classify loop (nb 04): move the probe over a grid, fire the glitch, read the target, map by X/Y.

Target: Arduino Uno (`arduino/uart_glitch_target`) — returns its count; ≠ 10000 means the glitch landed.

**Wiring:** GP0→D2, D3→1kΩ→GP1, GP2→RESET, common GND; Crowbar OUTPUT→Uno 5V (if `USE_CROWBAR`); EMFI coil on the chip.

> ⚠️ HV at every point. Shield on, nothing under the probe, small grid first. EMFI can reset the CNC's USB adapter — `CncStage` reconnects automatically.

## 1 · Place the probe and set the origin

Jog over the chip and **Set origin here** (`G92`). Adjust `CNC_PORT` if it re-enumerated.

In [ ]:
import time
import faultycat as fc

CNC_PORT, CNC_BAUD = "/dev/ttyUSB0", 115200   # check with list_ports if unsure
stage = fc.CncStage(CNC_PORT, CNC_BAUD)
stage

In [ ]:
fc.cnc_panel(stage)

### Work volume — separate interface

Define and **frame** the sweep region: set origin, capture corners or type bounds, **Frame volume** to trace it (returns to origin), and copy the `XS`/`YS`/`ZS` lines into the campaign.

In [ ]:
fc.cnc_volume_panel(stage)

## 2 · Connect the board and target

Open the board + target UART and set the target/engine params. The swept axes are the CNC coordinates.

In [ ]:
# --- target ---
SIM       = False              # True to dry-run without hardware
CMD       = b'\xAA'           # byte that makes the target run + report
EXPECTED  = b'10000'           # NORMAL answer (no fault)
BAUD      = 9600               # target UART baud

# --- engines (EMFI is the spatial one; crowbar optional) ---
USE_EMFI      = True           # EM glitch (coil moved by the CNC) — HV!
USE_CROWBAR   = False          # EMFI-only demo (set True + add 'width' axis for a crowbar sweep)
OUTPUT        = 'lp'           # crowbar path: 'lp' or 'hp'
EMFI_WIDTH_US = 14             # EMFI pulse width (us) — found to fault this target
WIDTHS_NS     = [500, 1000, 2000, 4000, 6000, 8000, 10000, 12000]  # crowbar widths SWEPT (ns)
PULSES        = 8              # glitch pulses per trigger (multipulse)
GAP_US        = 100            # inter-pulse gap (delay_us)

# Hardware target reset (clean state each try). GP2 -> Arduino RESET.
RST_GP    = 2                  # None to skip
RST_MS    = 10

cat = fc.connect(simulator=SIM)
cat.uart.open(baud=BAUD)
print('target says:', cat.uart.readline(timeout=1.5))   # its BOOT banner
cat

### Classify each attempt from the target's response

In [ ]:
def classify(line: bytes) -> str:
    s = line.strip()
    if s == EXPECTED:  return 'normal'    # target unfazed
    if not s:          return 'crash'     # no answer — hung
    if b'BOOT' in s:   return 'reset'     # rebooted mid-count
    return 'success'                      # corrupted count, survived — a clean fault

## 3 · Spatial attack sweep

Axes = CNC coordinates. `set_range` takes the values to visit (`range(0, 51)` for 0..50); keep it small. Each point: **move → reset → charge → CMD → fire → read → classify.**

In [ ]:
# XS, YS = [0.0, 1.0, 2.0], [0.0, 1.0, 2.0]   # probe grid over the chip (mm from origin)

XS = [round(0.0 + i*5.0, 3) for i in range(8)]   # 0..35 mm step 5
YS = [round(0.0 + i*5.0, 3) for i in range(2)]   # 0, 5 mm

gc = fc.GlitchController(["x", "y"],
                        groups=["success", "reset", "crash", "normal", "error"])
gc.set_range("x", XS)
gc.set_range("y", YS)

if USE_CROWBAR:
    cat.crowbar.trigger = 'immediate'; cat.crowbar.output = OUTPUT
    cat.crowbar.delay_us = GAP_US   # width_ns is set per-point in the loop
    cat.crowbar.repeat = PULSES
if USE_EMFI:
    cat.emfi.trigger = 'immediate'; cat.emfi.delay_us = GAP_US
    cat.emfi.width_us = EMFI_WIDTH_US; cat.emfi.repeat = PULSES

fired = ' + '.join(e for e, on in [('crowbar', USE_CROWBAR), ('emfi', USE_EMFI)] if on)
print(f"firing: {fired or 'NOTHING — enable USE_EMFI and/or USE_CROWBAR'}")
print(f"grid: {len(XS)}x{len(YS)} = {len(XS) * len(YS)} points (EMFI {EMFI_WIDTH_US}us/{PULSES}p)")

Run the sweep:

In [ ]:
emfi_s = f"{EMFI_WIDTH_US}us/{PULSES}p" if USE_EMFI else "-"      # width / pulses
print(f"{'x':>6} {'y':>6}  {'emfi':>9} {'crowbar':>10}  outcome   reply")
for p in gc.glitch_values():
    fc.cnc_move_to(stage, p, feed=1500)          # position the probe (nb 06)
    if USE_CROWBAR:
        cat.crowbar.width_ns = p["width"]        # swept crowbar pulse width (ns)
    cb_s = f"{p['width']}ns/{PULSES}p" if USE_CROWBAR else "-"
    if RST_GP is not None:
        cat.target_reset(RST_GP, RST_MS)         # clean known state each try
        time.sleep(2.0)                          # let the Uno boot to idle (~1.8 s)
    if USE_EMFI:                                 # charge HV BEFORE sending CMD
        cat.emfi.arm(); cat.emfi.wait_for_charged()
    if USE_CROWBAR:
        cat.crowbar.arm()
    cat.uart.reset_input()                       # drop stale bytes / BOOT banner
    cat.uart.write(CMD)                          # target starts its ~45 ms count
    time.sleep(0.003)
    try:
        if USE_CROWBAR: cat.crowbar.fire()       # voltage ...
        if USE_EMFI:    cat.emfi.fire()          # ... then EM (same window)
        if USE_CROWBAR: cat.crowbar.disarm()
        if USE_EMFI:    cat.emfi.disarm()
    except fc.EngineError:
        gc.add('error', emfi=emfi_s, crowbar=cb_s)
        print(f"{p['x']:>6} {p['y']:>6}  {emfi_s:>9} {cb_s:>10}  error"); continue
    line = cat.uart.read_until(b'\n', timeout=2.5)   # long enough to catch a reset's BOOT banner (~1.8s reboot)
    res = classify(line)
    gc.add(res, emfi=emfi_s, crowbar=cb_s)       # record engine params with the point
    print(f"{p['x']:>6} {p['y']:>6}  {emfi_s:>9} {cb_s:>10}  {res:<8}  {line!r}")
    if RST_GP is None and res in ('crash', 'reset'):
        time.sleep(1.5)                          # no reset wire: wait out the reboot

stage.goto_origin(1500)   # back to the work origin (home) when the sweep ends
gc.counts()

## Results

Vectors `x`, `y` (+ `emfi`/`crowbar`) and outcome `group`. Views:

- **2D map (X vs Y)** by outcome — the main spatial map.
- **Effect-rate heatmap** — fraction of effects per cell (with repeats).
- **3D map** — when Z is swept.
- **Interactive** (plotly, `[interactive]` extra).

In [ ]:
df = gc.results_df()
fc.glitch_map(df, "group", x="x", y="y", title="Spatial fault map — X vs Y");

**Effect-rate heatmap** — fraction of attempts per cell that were not `normal`.

In [ ]:
import matplotlib.pyplot as plt

df["effect"] = (df["group"] != "normal").astype(float)   # your definition of "something happened"
grid = df.pivot_table(index="y", columns="x", values="effect", aggfunc="mean")
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(grid.values, origin="lower", aspect="auto", cmap="magma",
               extent=[grid.columns.min(), grid.columns.max(),
                       grid.index.min(), grid.index.max()])
ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)"); ax.set_title("Effect rate over the chip")
fig.colorbar(im, ax=ax, label="fraction with an effect");

**3D map** — only if Z was swept.

In [ ]:
if "z" in df.columns:
    fc.glitch_map_3d(df, "group", x="x", y="y", z="z", title="Spatial fault map (3D)")
else:
    print("No 'z' column — add 'z' to fc.GlitchController([...]) for a 3D map.")

**Interactive** (plotly). Needs the `[interactive]` extra; uncomment:

In [ ]:
# fc.glitch_map_plotly(df, "group", x="x", y="y")

## Clean up

In [ ]:
stage.goto_origin()
cat.uart.close()
cat.close()
stage.close()